# MRI II — Oral Exam: Cartesian GRE Sequence Assembly

**Time limit: 30 minutes**

All building blocks (RF pulse, gradients, ADC, spoiler, timing) are already
constructed for you below. **Your only task is to assemble the sequence** —
i.e. write the loop that calls `seq.add_block()` in the correct order and
structure for this trajectory. Once correct, the rest of the notebook
(timing check, simulation, reconstruction) will run and produce an image —
that image is your feedback.



In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch


In [ ]:
seq_filename = "gre_cartesian_exam.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128

flip_angle_deg = 40
tr = 5000e-3
te = 5e-3

torch.set_default_device("cpu")


## System and Sequence Object

`system` holds the scanner hardware limits. `seq` is the container you add
blocks (RF pulses, gradients, delays, ADC windows) to, in chronological order.
Objects passed together to a single `seq.add_block()` call play *simultaneously*.


In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)


## Building Blocks (given)

- `rf` — sinc excitation pulse
- `gx` — readout gradient (flat_area spans the full $k_x$ line)
- `gx_pre` — readout pre-phaser (moves to $-k_{x,\text{max}}$)
- `adc` — readout window, aligned to `gx`'s flat top
- `gx_spoiler` — dephases residual transverse magnetization after each line
- `te_delay`, `tr_delay` — computed so the sequence hits the requested TE/TR


In [ ]:
delta_kx = 1 / fov_x
delta_ky = 1 / fov_y
raster_eps = 1e-12

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

gx = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=3.2e-3,
    system=system,
)

gx_pre = pp.make_trapezoid(
    channel="x",
    area=-gx.area / 2,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=gx.flat_time,
    delay=gx.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * gx.area,
    system=system,
)

te_delay = (
    te
    - (pp.calc_duration(rf) - pp.calc_rf_center(rf)[0])
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx) / 2
)

tr_delay = (
    tr
    - pp.calc_duration(rf)
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx)
    - pp.calc_duration(gx_spoiler)
    - te_delay
)

te_delay = np.ceil((te_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
tr_delay = np.ceil((tr_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time

assert np.all(te_delay >= 0)
assert np.all(tr_delay >= 0)


## Task: Assemble the Cartesian GRE Sequence

A Cartesian GRE acquires k-space **line by line**, from $-k_{y,\text{max}}$ to
$+k_{y,\text{max}}$, one phase-encoding step per TR.

**For each of the `n_y` phase-encoding steps:**
1. Add the RF pulse.
2. Build `gy_pre` for this step: `channel="y"`, `area=phase_area`,
   `duration=pp.calc_duration(gx_pre)` (must fit in the same time slot as `gx_pre`).
3. Add `gx_pre` and `gy_pre` together.
4. Add the TE delay.
5. Add `gx` and `adc` together.
6. Add `gx_spoiler`.
7. Add the TR delay.

**Hint:** the phase-encoding areas run from $-n_y \Delta k_y / 2$ to
$+n_y \Delta k_y / 2$ in `n_y` steps — use `np.linspace(..., endpoint=False)`.


In [ ]:
raise NotImplementedError


## Verification, Simulation, Reconstruction (given)

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot()

seq.write(seq_filename, v141_compat=True)


In [ ]:
phantom = mr0.util.load_phantom(size=(n_x, n_y))

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)


## Reconstruction (given)

Standard Cartesian inverse FFT — this is the same recon you built in Exercise 1.


In [ ]:
signal = signal.numpy()
kspace = signal.reshape((n_y, n_x))

# Necessary to fix orientation
kspace = kspace[::-1, ::-1]

image = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace)))

plt.figure()
plt.imshow(np.abs(image), cmap="gray")
plt.title("Reconstructed Cartesian GRE")
plt.show()
